# Backpropagation for a Dense Layer — Derivation and Demo

This notebook derives the backward pass for a Dense (fully-connected) layer from first principles and demonstrates a tiny training run. It then reproduces a small convergence experiment and exports figures (loss curve, gradient heatmap, and convergence comparison) into `notebooks/figures/` for use in the README or other documents.

## 1. Notation and forward pass

Let $x\in\mathbb{R}^{d\times N}$ be a batch of N column-inputs (each column is one example). A Dense layer has weights $W\in\mathbb{R}^{m\times d}$ and biases $b\in\mathbb{R}^{m\times 1}$. The layer computes:

$$z = W x + b$$

where $z\in\mathbb{R}^{m\times N}$ is the pre-activation output for the batch. An element-wise activation $\phi$ (e.g., ReLU, sigmoid) yields $a=\phi(z)$. For simplicity we use mean-squared error (MSE) for a scalar loss aggregated over the batch: $L(\theta)=\frac{1}{N}\sum_{i=1}^N \|\hat y_i - y_i\|^2$.

## 2. Backpropagation derivation (Dense layer)

We want $\partial L/\partial W$ and $\partial L/\partial b$. Using the chain rule, if we have upstream gradient $\partial L/\partial z$ (shape $m\times N$), then for the batch:

$$\frac{\partial L}{\partial W} = (\partial L/\partial z) x^T, \qquad \frac{\partial L}{\partial b} = \sum_{i=1}^N \partial L/\partial z_{:,i}$$

The gradient w.r.t. the input (to propagate to previous layers) is:
$$\frac{\partial L}{\partial x} = W^T (\partial L/\partial z).$$

This exact computation is implemented in `core.layers.Dense.backward`. The notebook demo below uses those functions directly to illustrate the numerics and to check correctness with numerical differentiation.

In [ ]:
# Import project modules (adjust path if running from repo root)
import sys
from pathlib import Path
PROJ = Path('..').resolve()
sys.path.insert(0, str(PROJ))
import numpy as np
from core.layers import Dense
from core.network import NeuralNetwork
from utils.plotting import plot_loss, plot_gradient_heatmap, plot_convergence_comparison
import matplotlib.pyplot as plt
print('imports ok')

In [ ]:
# Numerical finite-difference check for Dense.backward (small example)
rng = np.random.default_rng(0)
in_f, out_f = 3, 2
dense = Dense(in_f, out_f, init_scheme='normal', seed=0)
x = rng.normal(size=(in_f, 5))
# scalar loss f(params) = mean((W x + b)^2) to test gradients
def loss_flat(params):
    Wf = params[:out_f*in_f].reshape((out_f,in_f))
    bf = params[out_f*in_f:].reshape((out_f,1))
    y = Wf @ x + bf
    return float(np.mean(y**2))
params = np.concatenate([dense.W.flatten(), dense.b.flatten()])
eps = 1e-6
num_grad = np.zeros_like(params)
for i in range(params.size):
    p = params.copy()
    p[i] += eps
    f1 = loss_flat(p)
    p[i] -= 2*eps
    f2 = loss_flat(p)
    num_grad[i] = (f1 - f2) / (2*eps)
# analytic gradient via Dense.backward (set x to cache)
dense.x = x
_ = dense.forward(x)
grad_out = 2 * dense.forward(x) / x.shape[1]
_ = dense.backward(grad_out, lr=0.0)
analytic = np.concatenate([dense.dW.flatten(), dense.db.flatten()])
rel_err = np.max(np.abs(num_grad - analytic))
print('max abs difference (num vs analytic):', rel_err)

The notebook above compares the numerical gradient (finite differences) to the analytic gradient computed by `Dense.backward`. A small difference indicates the backward implementation is consistent with numerical derivatives.

In [ ]:
# Tiny training demo and figure export
rng = np.random.default_rng(1)
X = rng.normal(size=(3, 200))
true_w = rng.normal(size=(1,3))
y = (true_w @ X).reshape(1, -1)
net = NeuralNetwork(layer_sizes=[3, 16, 1], activation='relu', init_scheme='he', seed=0)
net.train(X, y, lr=0.05, epochs=120, verbose=False)
fig1 = plot_loss(net.loss_history)
fig2 = plot_gradient_heatmap(net.grad_norms_per_epoch)
# ensure figures directory exists inside notebooks/
from pathlib import Path
fig_dir = Path('figures')  # when running notebook from notebooks/ folder this is ./notebooks/figures
fig_dir.mkdir(parents=True, exist_ok=True)
fig1_path = fig_dir / 'loss_curve.png'
fig2_path = fig_dir / 'grad_heatmap.png'
fig1.savefig(fig1_path, bbox_inches='tight')
fig2.savefig(fig2_path, bbox_inches='tight')
print('Saved', fig1_path, fig2_path)

In [ ]:
# Reproduce a small convergence experiment (activation vs init) and export
from experiments.run_convergence import simple_compare
acts = ['relu', 'gelu']
inits = ['he', 'xavier']
res = simple_compare(X, y, [3, 16, 1], acts, inits, lr=0.02, epochs=120)
fig3 = plot_convergence_comparison(res)
fig3_path = fig_dir / 'convergence_comparison.png'
fig3.savefig(fig3_path, bbox_inches='tight')
print('Saved', fig3_path)